# C3 — Manifestes de CREMA-D et de RAVDESS (P1, P2, P9)

Ce notebook trouve les audios des deux jeux d'émotions, construit leur manifeste avec le code du
dépôt (`echolens.core.data.cremad` et `echolens.core.data.ravdess`), puis vérifie les effectifs
attendus. Il n'écrit que deux CSV de métadonnées, `cremad.csv` et `ravdess.csv`, sans aucun
audio (I8). ffmpeg n'est pas nécessaire : la durée est lue dans l'en-tête des WAV.

## Démarrage

On clone le dépôt (branche `BRANCH`) et on l'installe en mode éditable (I9), puis on ajoute
`src/` au `sys.path` : l'installation éditable n'est visible qu'après un redémarrage du noyau.

In [ ]:
import sys

BRANCH = "main"
REPO_URL = "https://github.com/mentalist011101/TP_SIGNAL_DS.git"
CLONE_DIR = "/kaggle/working/TP_SIGNAL_DS"

!rm -rf {CLONE_DIR}
!git clone --depth 1 --branch {BRANCH} {REPO_URL} {CLONE_DIR}
# Kaggle tourne sous Python 3.13, au-delà de requires-python (<3.13) du dépôt.
%pip install -q --ignore-requires-python -e {CLONE_DIR}

sys.path.insert(0, CLONE_DIR + "/src")
print(sys.version)

## a. Sources

On cherche d'abord les copies Kaggle attachées au kernel (`/kaggle/input`) :

- CREMA-D : un dossier `AudioWAV` qui contient exactement 7 442 WAV ;
- RAVDESS : un dossier qui contient `Actor_01` … `Actor_24`, soit exactement 1 440 WAV de parole.
  Certaines copies contiennent les fichiers **en double** (deux dossiers) : on prend un dossier
  où chaque fichier n'apparaît qu'une fois, et on ne compte jamais les doublons.

Si aucune copie ne convient, on télécharge depuis les sources officielles : le dépôt GitHub de
CREMA-D (fichiers Git LFS, servis par media.githubusercontent.com) et l'archive de RAVDESS sur
Zenodo (records/1188976).

In [ ]:
from pathlib import Path

CREMAD_EXPECTED_WAV = 7442
RAVDESS_EXPECTED_WAV = 1440
INPUT_DIR = Path("/kaggle/input")


def find_cremad_root() -> Path | None:
    # Racine = dossier parent d'un AudioWAV qui contient exactement 7 442 WAV.
    for audio_folder in sorted(INPUT_DIR.glob("**/AudioWAV")):
        wav_count = len(list(audio_folder.glob("*.wav")))
        print("CREMA-D candidat :", audio_folder, wav_count, "WAV")
        if wav_count == CREMAD_EXPECTED_WAV:
            return audio_folder.parent
    return None


def find_ravdess_root() -> Path | None:
    # Racine = dossier qui contient Actor_01 … Actor_24, chaque nom de fichier une seule fois.
    candidates = sorted({actor.parent for actor in INPUT_DIR.glob("**/Actor_01")})
    for candidate in candidates:
        wav_paths = list(candidate.glob("Actor_*/*.wav"))
        unique_names = {path.name for path in wav_paths}
        print("RAVDESS candidat :", candidate, len(wav_paths), "WAV,", len(unique_names), "noms")
        if len(wav_paths) == RAVDESS_EXPECTED_WAV and len(unique_names) == RAVDESS_EXPECTED_WAV:
            return candidate
    return None


cremad_root = find_cremad_root()
ravdess_root = find_ravdess_root()
print("CREMA-D :", cremad_root)
print("RAVDESS :", ravdess_root)

### Repli : sources officielles

Exécuté seulement si une copie manque. CREMA-D : liste des fichiers par l'API GitHub, puis
téléchargement de chaque WAV depuis media.githubusercontent.com (Git LFS), 8 à la fois.
RAVDESS : archive `Audio_Speech_Actors_01-24.zip` de Zenodo.

In [ ]:
import json
import urllib.request
import zipfile
from concurrent.futures import ThreadPoolExecutor

DOWNLOAD_DIR = Path("/kaggle/working/downloads")
CREMAD_TREE_URL = (
    "https://api.github.com/repos/CheyneyComputerScience/CREMA-D/git/trees/master?recursive=1"
)
CREMAD_MEDIA_URL = (
    "https://media.githubusercontent.com/media/CheyneyComputerScience/CREMA-D/master/"
)
RAVDESS_ZIP_URL = (
    "https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip?download=1"
)
PARALLEL_DOWNLOADS = 8


def download_cremad_file(relative_path: str) -> None:
    destination = DOWNLOAD_DIR / "cremad" / relative_path
    if destination.exists():
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(CREMAD_MEDIA_URL + relative_path, destination)


def download_cremad() -> Path:
    with urllib.request.urlopen(CREMAD_TREE_URL) as response:
        tree = json.load(response)["tree"]
    wav_paths = []
    for entry in tree:
        path = entry["path"]
        if path.startswith("AudioWAV/") and path.endswith(".wav"):
            wav_paths.append(path)
    print("CREMA-D : téléchargement de", len(wav_paths), "fichiers")
    with ThreadPoolExecutor(max_workers=PARALLEL_DOWNLOADS) as pool:
        list(pool.map(download_cremad_file, wav_paths))
    return DOWNLOAD_DIR / "cremad"


def download_ravdess() -> Path:
    archive_path = DOWNLOAD_DIR / "ravdess.zip"
    archive_path.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(RAVDESS_ZIP_URL, archive_path)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(DOWNLOAD_DIR / "ravdess")
    archive_path.unlink()
    return DOWNLOAD_DIR / "ravdess"


if cremad_root is None:
    cremad_root = download_cremad()
if ravdess_root is None:
    ravdess_root = download_ravdess()
print("CREMA-D :", cremad_root)
print("RAVDESS :", ravdess_root)

## b. Genre des acteurs de CREMA-D

`VideoDemographics.csv` donne le genre (colonne `Sex`) de chaque acteur. On le télécharge
depuis le dépôt officiel dans `/kaggle/working` ; il n'est jamais commité (I8).

In [ ]:
DEMOGRAPHICS_URL = (
    "https://raw.githubusercontent.com/CheyneyComputerScience/CREMA-D/master/VideoDemographics.csv"
)
demographics_path = Path("/kaggle/working/VideoDemographics.csv")
urllib.request.urlretrieve(DEMOGRAPHICS_URL, demographics_path)
print(demographics_path.read_text(encoding="utf-8-sig").splitlines()[:3])

## c. Convention `<datasets_dir>/<id>`

Les manifestes donnent des chemins relatifs à `<datasets_dir>/<id du jeu>` (docs/datasets.md).
On crée donc `/kaggle/working/datasets/cremad` et `/kaggle/working/datasets/ravdess`, liens
symboliques vers les racines trouvées.

In [ ]:
from echolens.core.data.manifest import dataset_root

datasets_dir = Path("/kaggle/working/datasets")
datasets_dir.mkdir(parents=True, exist_ok=True)
for dataset_id, root in [("cremad", cremad_root), ("ravdess", ravdess_root)]:
    link = dataset_root(dataset_id, datasets_dir)
    if link.is_symlink():
        link.unlink()
    link.symlink_to(root)
    print(link, "→", root)

## d. Manifestes

Une ligne par fichier audio, étiquette d'origine gardée telle quelle (P9) : codes `ANG` … `SAD`
pour CREMA-D, `01` … `08` pour RAVDESS, calme (`02`) et surprise (`08`) compris.

In [ ]:
from echolens.core.data.cremad import build_cremad_manifest
from echolens.core.data.manifest import save_manifest
from echolens.core.data.ravdess import build_ravdess_manifest

cremad_rows = build_cremad_manifest(dataset_root("cremad", datasets_dir), demographics_path)
ravdess_rows = build_ravdess_manifest(dataset_root("ravdess", datasets_dir))
save_manifest(cremad_rows, Path("/kaggle/working/cremad.csv"))
save_manifest(ravdess_rows, Path("/kaggle/working/ravdess.csv"))
print("CREMA-D :", len(cremad_rows), "lignes")
print("RAVDESS :", len(ravdess_rows), "lignes")

## e. Contrôles d'effectifs

Chaque chiffre est comparé à la documentation des jeux. Au moindre écart, le notebook s'arrête
avec la liste des différences : un manifeste faux ne doit jamais partir vers l'entraînement.

In [ ]:
import statistics
from collections import Counter

from echolens.core.data import cremad, ravdess


def count_by(rows, field):
    counts = Counter()
    for row in rows:
        counts[getattr(row, field)] += 1
    return dict(sorted(counts.items()))


def actors_by_gender(rows):
    # Un acteur compte une fois, quel que soit son nombre de fichiers.
    gender_of_actor = {}
    for row in rows:
        gender_of_actor[row.speaker] = row.gender
    counts = Counter(gender_of_actor.values())
    return dict(sorted(counts.items()))


def compare(label, observed, expected, problems):
    print(f"{label} : {observed}")
    if observed != expected:
        problems.append(f"{label} : obtenu {observed}, attendu {expected}")


def describe_durations(name, rows):
    durations = [row.duration_s for row in rows]
    print(
        f"{name} : durée min {min(durations):.3f} s, "
        f"médiane {statistics.median(durations):.3f} s, max {max(durations):.3f} s"
    )


problems = []

# CREMA-D : 7 442 fichiers moins les éventuels défectueux connus.
cremad_bad_by_label = Counter()
for bad_name in cremad.KNOWN_BAD_FILES:
    bad_label = bad_name.split("_")[2]
    cremad_bad_by_label[bad_label] += 1
cremad_expected_labels = {}
for code in ["ANG", "DIS", "FEA", "HAP", "SAD"]:
    cremad_expected_labels[code] = 1271 - cremad_bad_by_label[code]
cremad_expected_labels["NEU"] = 1087 - cremad_bad_by_label["NEU"]
compare("CREMA-D lignes", len(cremad_rows), 7442 - len(cremad.KNOWN_BAD_FILES), problems)
compare("CREMA-D acteurs", len(count_by(cremad_rows, "speaker")), 91, problems)
compare("CREMA-D acteurs par genre", actors_by_gender(cremad_rows), {"F": 43, "M": 48}, problems)
compare(
    "CREMA-D extraits par émotion",
    count_by(cremad_rows, "label"),
    dict(sorted(cremad_expected_labels.items())),
    problems,
)

# RAVDESS : 1 440 fichiers, 24 acteurs de 60 fichiers.
ravdess_expected_labels = {"01": 96}
for code in ["02", "03", "04", "05", "06", "07", "08"]:
    ravdess_expected_labels[code] = 192
compare("RAVDESS lignes", len(ravdess_rows), 1440 - len(ravdess.KNOWN_BAD_FILES), problems)
ravdess_files_per_actor = count_by(ravdess_rows, "speaker")
compare("RAVDESS acteurs", len(ravdess_files_per_actor), 24, problems)
compare("RAVDESS fichiers par acteur", set(ravdess_files_per_actor.values()), {60}, problems)
compare("RAVDESS acteurs par genre", actors_by_gender(ravdess_rows), {"F": 12, "M": 12}, problems)
compare(
    "RAVDESS extraits par émotion",
    count_by(ravdess_rows, "label"),
    ravdess_expected_labels,
    problems,
)

describe_durations("CREMA-D", cremad_rows)
describe_durations("RAVDESS", ravdess_rows)

if problems:
    raise RuntimeError(
        "Effectifs inattendus, manifestes à ne pas utiliser :\n" + "\n".join(problems)
    )
print("Tous les effectifs sont conformes.")

## Nettoyage

On supprime le clone du dépôt, les liens et un éventuel téléchargement : les sorties ne gardent
que les deux manifestes (le CSV démographique est aussi retiré, I8).

In [ ]:
import shutil

shutil.rmtree(CLONE_DIR, ignore_errors=True)
shutil.rmtree(datasets_dir, ignore_errors=True)
shutil.rmtree(DOWNLOAD_DIR, ignore_errors=True)
demographics_path.unlink(missing_ok=True)
remaining = [path.name for path in Path("/kaggle/working").iterdir()]
print(sorted(remaining))